# Full GPU experiments on Colab (CUDA + OpenMP + MPI)

This notebook repeats every GPU experiment in the report and takes much longer than `colab_demo.ipynb`. For the live demo, use `colab_demo.ipynb` instead.

1. **Runtime > Change runtime type > T4 GPU**
2. Nothing else to set up: the repository is public. (A `GH_TOKEN` Colab secret is only needed if the repository is ever made private again.)
3. Run the cells top to bottom (or **Runtime > Run all**).
4. The last cell downloads `results.zip` with all CSVs, plots and profiler reports.

Colab gives only 2 CPU threads, so the OpenMP scaling experiments are better run on a laptop or lab machine (see README). This notebook covers correctness, the GPU experiments and profiling.

In [ ]:
# Branch to test.
BRANCH = "main"
!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv

In [ ]:
import os, subprocess

def sh(cmd, secret=None):
    """Run a shell command, show its output, and stop the notebook if it fails."""
    hide = (lambda t: t.replace(secret, "***")) if secret else (lambda t: t)
    print("$", hide(cmd))
    r = subprocess.run("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(hide(r.stdout[-4000:]))
    if r.returncode != 0:
        raise RuntimeError(f"command failed (exit {r.returncode}): {hide(cmd)}")

os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"          # Colab runs as root
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
sh("apt-get -qq update && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null && mpirun --version | head -1")
os.chdir("/content")
sh("rm -rf /content/lob")
# The repo is private: add a GitHub token as a Colab secret named GH_TOKEN
# (key icon in the left sidebar, then turn on "Notebook access").
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
repo = "github.com/abhisaarguptaa/Stock-market-simulator-using-PCAP.git"
url = f"https://x-access-token:{token}@{repo}" if token else f"https://{repo}"
if not token:
    print("No GH_TOKEN secret found: cloning without a token (works only if the repo is public).")
sh(f"git clone -b {BRANCH} {url} /content/lob", secret=token)
os.chdir("/content/lob")
sh("git log --oneline -1 && ls")
print("Setup OK, working in", os.getcwd())

## Build and unit tests

In [ ]:
os.chdir("/content/lob")
sh("cmake -S . -B build -DLOB_USE_CUDA=ON -DLOB_USE_MPI=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null && cmake --build build -j 2>&1 | tail -3")
sh("./build/lob_tests | tail -8")

## Correctness: every back end must produce the identical trade log

In [ ]:
%cd /content/lob
!bash scripts/check_determinism.sh build

## Model realism: stylised facts of returns

In [ ]:
%cd /content/lob
!mkdir -p results
!./build/lobsim --agents 64000 --instruments 16 --ticks 20000 --decide cuda --match omp --prices results/prices.csv --header
!python analysis/stylized_facts.py results/prices.csv --out results/facts
from IPython.display import Image, display
display(Image("results/facts/stylized_facts.png"))

## GPU experiments (CPU-vs-GPU crossover, growing agents, kernel ablations)
This is the longest step. Lower `TICKS` to make it faster.

In [ ]:
%cd /content/lob
!TICKS=200 REPS=3 bash scripts/bench.sh gpu build

## Profiling with Nsight Systems and Nsight Compute
If Nsight Compute reports `ERR_NVGPUCTRPERM`, GPU counters are locked on this VM; the timeline (Nsight Systems) still works.

In [ ]:
%cd /content/lob
!bash scripts/profile.sh build
!python analysis/roofline.py results/profile/ncu_sorted.csv results/profile/ncu_unsorted.csv --out results/plots/roofline.png

## Plots and summary

In [ ]:
%cd /content/lob
!python analysis/scaling.py results
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("results/plots/*.png")):
    print(p); display(Image(p))
print(open("results/summary.md").read())

In [ ]:
%cd /content/lob
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")